In [1]:
# Step 1: Install scispacy and the model
#!pip install scispacy
#!pip install spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# Step 2: Import required libraries
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
import json

In [3]:
# Base API endpoint for ClinicalTrials.gov v2
#API_URL = "https://clinicaltrials.gov/api/v2/studies"

# Search parameters
#params = {
#    "query.term": "Mild depression",
#    "filter.interventionType": "Drug",
#    "pageSize": 100,  # Max per page
#    "pageToken": ""   # Will be used for pagination
#}

url = (
    "https://clinicaltrials.gov/api/v2/studies?"
    "query.term=Major%20Depression&"
    "filter.interventionType=Drug&"
    "pageSize=100"
)

# Storage for trial data
all_trials = []

while True:
    #response = requests.get(API_URL, params=params)
    response = requests.get(url)
    print(response)
    data = response.json()

    # Extract relevant fields
    for study in data.get("studies", []):
        protocol_section = study.get("protocolSection", {})
        identification_module = protocol_section.get("identificationModule", {})
        status_module = protocol_section.get("statusModule", {})
        design_module = protocol_section.get("designModule", {})
        intervention_module = protocol_section.get("interventionModule", {})
        sponsor_module = protocol_section.get("sponsorCollaboratorsModule", {})

        # Extract sponsor name
        sponsors = sponsor_module.get("sponsors", [])
        sponsor_name = sponsors[0].get("leadSponsor", {}).get("name") if sponsors else None

        # Extract primary outcome measures
        primary_outcomes = protocol_section.get("outcomesModule", {}).get("primaryOutcomes", [])
        primary_measures = "; ".join([po.get("measure") for po in primary_outcomes if po.get("measure")])

        # Extract reason for termination
        why_stopped = status_module.get("whyStopped")

        record = {
            "NCT ID": identification_module.get("nctId"),
            "Title": identification_module.get("briefTitle"),
            "Status": status_module.get("overallStatus"),
            "Phase": design_module.get("phase"),
            "Start Date": status_module.get("startDateStruct", {}).get("date"),
            "Drugs": ", ".join(
                i["interventionName"]
                for i in intervention_module.get("interventions", [])
                if i["interventionType"].lower() == "drug"
            ),
            "Sponsor": sponsor_name,
            "Primary Outcome Measures": primary_measures,
            "Reason for Termination": why_stopped
        }
        all_trials.append(record)

    # Pagination
    next_token = data.get("nextPageToken")
    if not next_token:
        break
    params["pageToken"] = next_token

# Convert to DataFrame
df = pd.DataFrame(all_trials)

# Save or display
#df.to_csv("mdd_drug_trials_enhanced.csv", index=False)
print(df.head())


<Response [400]>


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [170]:
import requests
import pandas as pd

# Construct the full URL with embedded query parameters

url = (
    "https://clinicaltrials.gov/api/v2/studies?"
    "query.term=Drug&"
    "query.cond=Major%20Depressive%20Disorder&"
    "pageSize=100"
)
# Send the GET request
response = requests.get(url)

df = pd.DataFrame(columns=["nct_id", "Study_title", "Study_status","Study_start_date"
                           ,"Study_lead_sponsor_or_company","Indications/Conditions","Drug_names"
                          ,"Primary_outcomes","Secondary_outcome","Eligibilty_criteria"])

# Check if the request was successful
if response.status_code == 200:
    json_text = response.text
    info = json.loads(json_text)
    j = 0 
    print(type(info["studies"]))
    for i in info["studies"]:
        j = j + 1
        print("-------------%d study--------------"%j)
        #print(i)
        nct_id = i['protocolSection']['identificationModule']['nctId']
        #print("NCT ID: %s" %nct_id)
        try:
            official_title = i['protocolSection']['identificationModule']['officialTitle']
            print("Study title: %s" %official_title)
        except: 
            continue
            
        try: 
            overall_status = i['protocolSection']['statusModule']['overallStatus']
            print("Study Status: %s" %overall_status)
        except:
            continue
            
        try: 
            study_start_date = i['protocolSection']['statusModule']['startDateStruct']['date']
            print("Study start date: %s" %study_start_date)
        except: 
            continue

        try:     
            lead_sponsor = i['protocolSection']['sponsorCollaboratorsModule']['leadSponsor']['name']
            print("Study Lead Sponsor/Company: %s" %lead_sponsor)
        except: 
            continue
        #conditions_list = (i['protocolSection']['conditionsModule']['conditions'])
        #conditions = "; ".join(conditions_list)
        #print(i['protocolSection']['armsInterventionsModule'])
        #print(conditions)
        #interventions_module = protocol_section.get("armsInterventionsModule", {}).get("interventions", {})
        conditions = []
        for x in i['derivedSection']['conditionBrowseModule']['meshes']: 
            #print(x['term'])
            if x['term'].lower not in conditions: 
                conditions.append(x['term'])
            else: 
                continue
        print("Indications/Conditions treated :%s" %conditions)
        #for x in i['protocolSection']:
        #    print(type(x))
        drug_names = []
        try: 
            value_parent =  i['protocolSection']['armsInterventionsModule']['interventions']
            #print(value_parent)
            for z in value_parent: 
                #print(type(i))
                if z['type']=='DRUG':
                    value = z['name']
                    drug_names.append(value)
                else:
                    continue
        except: 
            #print('No interventions found for %s' %nct_id)
            continue
        print("Drug names :%s" %drug_names)

        try: 
            value_parent_1 = i['protocolSection']['outcomesModule']['primaryOutcomes']
            for a_2 in value_parent_1: 
                #print(type(a_2))
                primary_outcomes = (";;; ".join(list(a_2.values())))    
        except: 
            continue
        print("Primary Outcomes: %s" %primary_outcomes)
    
        try: 
            value_parent_2 = i['protocolSection']['outcomesModule']['secondaryOutcomes']
            for a_3 in value_parent_2: 
                #print(type(a_2))
                secondary_outcomes = (";;; ".join(list(a_3.values())))    
        except: 
            continue
        print("Secondary Outcomes: %s" %secondary_outcomes)

        try: 
            value_parent_3 = i['protocolSection']['eligibilityModule']
            #print(value_parent_3)
            result = ";;; ".join(f"{key}: {value}" for key, value in value_parent_3.items())
            print("Eligibilty Criteria: %s" %result)
        except: 
            continue


        df.loc[len(df)] = [nct_id,official_title,overall_status,study_start_date,lead_sponsor,conditions
                       ,drug_names,primary_outcomes,secondary_outcomes,eligibility]

         
    #data = response.json()
    # Process the data as needed
    # For example, extract study titles
    #studies = data.get("studies", [])
    #titles = [study.get("protocolSection", {}).get("identificationModule", {}).get("briefTitle") for study in studies]
    # Create a DataFrame
    #df = pd.DataFrame(titles, columns=["Study Title"])
    #print(df.head())
else:
    print(f"Request failed with status code {response.status_code}")

<class 'list'>
-------------1 study--------------
Study title: A Six-Week, Fixed Dose, Double-Blind, Double-Dummy, Placebo and Sertraline Controlled, Multicentre Trial to Evaluate the Safety and Efficacy of CP-316,311 in Outpatients With Major Depressive Disorder
Study Status: TERMINATED
Study start date: 2005-04
Study Lead Sponsor/Company: Pfizer
Indications/Conditions treated :['Depressive Disorder', 'Depression', 'Depressive Disorder, Major']
Drug names :['CP-316,311', 'Placebo', 'Sertraline']
Primary Outcomes: Change from baseline in HAM-D (17) at the week 6 visit
Secondary Outcomes: Change from baseline in MADRS, HAM-A, CGI-S and CGI-I at the week 6 visit.
Eligibilty Criteria: eligibilityCriteria: Inclusion Criteria:

* Outpatients with DSM-IV major depressive disorder

Exclusion Criteria:

* Women of child bearing potential;;; healthyVolunteers: False;;; sex: ALL;;; minimumAge: 18 Years;;; stdAges: ['ADULT', 'OLDER_ADULT']
-------------2 study--------------
Study title: A Double-

In [171]:
df

,nct_id,Study_title,Study_status,Study_start_date,Study_lead_sponsor_or_company,Indications/Conditions,Drug_names,Primary_outcomes,Secondary_outcome,Eligibilty_criteria
0,NCT00143091,"A Six-Week, Fixed Dose, Double-Blind, Double-D...",TERMINATED,2005-04,Pfizer,"[Depressive Disorder, Depression, Depressive D...","[CP-316,311, Placebo, Sertraline]",Change from baseline in HAM-D (17) at the week...,"Change from baseline in MADRS, HAM-A, CGI-S an...",eligibilityCriteria: Inclusion Criteria:\n\n* ...
1,NCT00048594,"A Double-Blind, Placebo-Controlled, Multicente...",COMPLETED,2002-01-03,Merck Sharp & Dohme LLC,"[Depressive Disorder, Depression, Depressive D...","[aprepitant, Comparator: placebo (unspecified)]",Total HAMD-17 score </= 10 after 10 weeks of t...,CGI-S score change from Week 10 at Month 6; HA...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
2,NCT05165394,"A Randomized, Double-Blind, Placebo-Controlled...",COMPLETED,2021-11-30,Neurocrine Biosciences,"[Anhedonia, Depressive Disorder, Depression, D...","[Placebo, NBI-1065846]","Change in Anhedonia Severity, as Measured by C...","CGI-S Scores;;; The CGI-S scale, which is base...",eligibilityCriteria: Inclusion Criteria:\n\n* ...
3,NCT06228391,Examining the Efficacy and Safety of Subanesth...,NOT_YET_RECRUITING,2024-03,Minneapolis Veterans Affairs Medical Center,"[Brain Injuries, Brain Injuries, Traumatic, De...","[IV ketamine, Midazolam]",Change in Severity of Depressive Symptoms;;; T...,Change in Severity of Post-traumatic Stress Di...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
4,NCT00866294,"A Randomized, Double-blind, Placebo Controlled...",COMPLETED,2009-04,GlaxoSmithKline,"[Depressive Disorder, Depression, Depressive D...","[paroxetine IR 10mg tablet, paroxetine IR 20mg...",Adjusted Mean Change From Baseline in the Hami...,Percentage of Responders Based on the Clinical...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
...,...,...,...,...,...,...,...,...,...,...
61,NCT03322774,Sleep To Reduce Incident Depression Effectively,COMPLETED,2018-03-09,Henry Ford Health System,"[Sleep Initiation and Maintenance Disorders, D...",[],Mediation of Depression Prevention by Reducing...,Stepped care insomnia treatment of dCBT-I and ...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
62,NCT05431374,A Randomized Controlled Trial of Measurement-B...,COMPLETED,2022-09-01,Pakistan Institute of Living and Learning,"[Depressive Disorder, Depression, Depressive D...",[],Hamilton Depression Rating (HDRS-17) - Time to...,All-cause discontinuation;;; Discontinuation f...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
63,NCT03870776,"Double-blind,Controlled,Randomized Phase 2 Stu...",ACTIVE_NOT_RECRUITING,2019-06-01,Mapreg,[Depression],"[MAP4343, Placebo]",Hamilton Depression Rating Scale score evoluti...,Blood quantification of gut microbiome;;; Meta...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
64,NCT06937476,Investigation of the Neurobiological Mechanism...,NOT_YET_RECRUITING,2025-05,Central South University,"[Rumination Syndrome, Depressive Disorder, Dep...","[Escitalopram, Aripiprazole 5mg]",Change in Ruminative Responses Scale (RRS) sco...,Change in 24-item Hamilton Depression Rating S...,eligibilityCriteria: Inclusion Criteria:\n\n* ...


In [18]:
# API endpoint for Major Depressive Disorder
#url = "https://clinicaltrials.gov/api/v2/studies?query.cond='Major%20Depressive%20Disorder'"
new_url = "https://clinicaltrials.gov/api/v2/studies?query.term=Drug&query.cond=Major%20Depressive%20Disorder"

# Make the GET request
response = requests.get(new_url)

# Check if request was successful
if response.status_code == 200:
    #json_data = response.json()
    json_text = response.text

    info = json.loads(json_text)

In [19]:
type(info["studies"])

list

In [20]:
#print(info["studies"])

In [168]:
j = 0 
df = pd.DataFrame(columns=["nct_id", "Study_title", "Study_status","Study_start_date"
                           ,"Study_lead_sponsor_or_company","Indications/Conditions","Drug_names"
                          ,"Primary_outcomes","Secondary_outcome","Eligibilty_criteria"])
for i in info["studies"]:
    #if i['protocolSection']['armsInterventionsModule']['interventions']['type'] == 'DRUG':        
    j = j + 1
    print("-------------%d study--------------"%j)
    #print(i)
    nct_id = i['protocolSection']['identificationModule']['nctId']
    official_title = i['protocolSection']['identificationModule']['officialTitle']
    overall_status = i['protocolSection']['statusModule']['overallStatus']
    study_start_date = i['protocolSection']['statusModule']['startDateStruct']['date']
    lead_sponsor = i['protocolSection']['sponsorCollaboratorsModule']['leadSponsor']['name']
    #conditions_list = (i['protocolSection']['conditionsModule']['conditions'])
    #conditions = "; ".join(conditions_list)
    #print(i['protocolSection']['armsInterventionsModule'])
    print("NCT ID: %s" %nct_id)
    print("Study title: %s" %official_title)
    print("Study Status: %s" %overall_status)
    print("Study start date: %s" %study_start_date)
    print("Study Lead Sponsor/Company: %s" %lead_sponsor)
    #print(conditions)
    
    #interventions_module = protocol_section.get("armsInterventionsModule", {}).get("interventions", {})
    conditions = []
    for x in i['derivedSection']['conditionBrowseModule']['meshes']: 
        #print(x['term'])
        if x['term'].lower not in conditions: 
            conditions.append(x['term'])
        else: 
            continue
    print("Indications/Conditions treated :%s" %conditions)
    #for x in i['protocolSection']:
    #    print(type(x))
    drug_names = []
    try: 
        value_parent =  i['protocolSection']['armsInterventionsModule']['interventions']
        #print(value_parent)
        for z in value_parent: 
            #print(type(i))
            if z['type']=='DRUG':
                value = z['name']
                drug_names.append(value)
            else:
                continue
    except: 
        print('No interventions found for %s' %nct_id)
    print("Drug names :%s" %drug_names)
    #print(i['protocolSection']['outcomesModule'])
    
    try: 
        value_parent_1 = i['protocolSection']['outcomesModule']['primaryOutcomes']
        for a_2 in value_parent_1: 
            #print(type(a_2))
            primary_outcomes = (";;; ".join(list(a_2.values())))    
    except: 
        continue
    print("Primary Outcomes: %s" %primary_outcomes)
    
    try: 
        value_parent_2 = i['protocolSection']['outcomesModule']['secondaryOutcomes']
        for a_3 in value_parent_2: 
            #print(type(a_2))
            secondary_outcomes = (";;; ".join(list(a_3.values())))    
    except: 
        continue
    print("Secondary Outcomes: %s" %secondary_outcomes)

    print('----------------------------------')

    try: 
        value_parent_3 = i['protocolSection']['eligibilityModule']
        #print(value_parent_3)
        eligibility = ";;; ".join(f"{key}: {value}" for key, value in value_parent_3.items())
        print("Eligibilty Criteria: %s" %eligibility)
        #for a_2 in value_parent_1: 
            #print(type(a_2))
        #    primary_outcomes = (";;; ".join(list(a_2.values())))    
    except: 
        continue
    #print("Primary Outcomes: %s" %primary_outcomes)
    
    #print(x)
    #print(interventions_module)
    #for i in interventions_module: 
    #    print('-----')
    #    print(i)
    #else: 
    #    continue

    df.loc[len(df)] = [nct_id,official_title,overall_status,study_start_date,lead_sponsor,conditions
                       ,drug_names,primary_outcomes,secondary_outcomes,eligibility]

-------------1 study--------------
NCT ID: NCT00143091
Study title: A Six-Week, Fixed Dose, Double-Blind, Double-Dummy, Placebo and Sertraline Controlled, Multicentre Trial to Evaluate the Safety and Efficacy of CP-316,311 in Outpatients With Major Depressive Disorder
Study Status: TERMINATED
Study start date: 2005-04
Study Lead Sponsor/Company: Pfizer
Indications/Conditions treated :['Depressive Disorder', 'Depression', 'Depressive Disorder, Major']
Drug names :['CP-316,311', 'Placebo', 'Sertraline']
Primary Outcomes: Change from baseline in HAM-D (17) at the week 6 visit
Secondary Outcomes: Change from baseline in MADRS, HAM-A, CGI-S and CGI-I at the week 6 visit.
----------------------------------
Eligibilty Criteria: eligibilityCriteria: Inclusion Criteria:

* Outpatients with DSM-IV major depressive disorder

Exclusion Criteria:

* Women of child bearing potential;;; healthyVolunteers: False;;; sex: ALL;;; minimumAge: 18 Years;;; stdAges: ['ADULT', 'OLDER_ADULT']
-------------2 st

KeyError: 'officialTitle'

In [169]:
df

,nct_id,Study_title,Study_status,Study_start_date,Study_lead_sponsor_or_company,Indications/Conditions,Drug_names,Primary_outcomes,Secondary_outcome,Eligibilty_criteria
0,NCT00143091,"A Six-Week, Fixed Dose, Double-Blind, Double-D...",TERMINATED,2005-04,Pfizer,"[Depressive Disorder, Depression, Depressive D...","[CP-316,311, Placebo, Sertraline]",Change from baseline in HAM-D (17) at the week...,"Change from baseline in MADRS, HAM-A, CGI-S an...",eligibilityCriteria: Inclusion Criteria:\n\n* ...
1,NCT00048594,"A Double-Blind, Placebo-Controlled, Multicente...",COMPLETED,2002-01-03,Merck Sharp & Dohme LLC,"[Depressive Disorder, Depression, Depressive D...","[aprepitant, Comparator: placebo (unspecified)]",Total HAMD-17 score </= 10 after 10 weeks of t...,CGI-S score change from Week 10 at Month 6; HA...,eligibilityCriteria: Patients with Major Depre...
2,NCT05165394,"A Randomized, Double-Blind, Placebo-Controlled...",COMPLETED,2021-11-30,Neurocrine Biosciences,"[Anhedonia, Depressive Disorder, Depression, D...","[Placebo, NBI-1065846]","Change in Anhedonia Severity, as Measured by C...","CGI-S Scores;;; The CGI-S scale, which is base...",eligibilityCriteria: Key Inclusion Criteria:\n...
3,NCT06228391,Examining the Efficacy and Safety of Subanesth...,NOT_YET_RECRUITING,2024-03,Minneapolis Veterans Affairs Medical Center,"[Brain Injuries, Brain Injuries, Traumatic, De...","[IV ketamine, Midazolam]",Change in Severity of Depressive Symptoms;;; T...,Change in Severity of Post-traumatic Stress Di...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
4,NCT00866294,"A Randomized, Double-blind, Placebo Controlled...",COMPLETED,2009-04,GlaxoSmithKline,"[Depressive Disorder, Depression, Depressive D...","[paroxetine IR 10mg tablet, paroxetine IR 20mg...",Adjusted Mean Change From Baseline in the Hami...,Percentage of Responders Based on the Clinical...,eligibilityCriteria: Inclusion Criteria:\n\n\<...
5,NCT01784991,"""Does a 1 Plus 1 Hydromorphone Pain Protocol H...",TERMINATED,2011-12,Albert Einstein Healthcare Network,"[Respiratory Insufficiency, Depression, Depres...","[Hydromorphone, Usual care group]",Respiratory Depression;;; Incidence of respira...,Serious Adverse Events;;; Defined as prolonged...,eligibilityCriteria: Inclusion Criteria:\n\n1....
6,NCT00361491,"An Eight-Week, Multicenter, Double-Blind, Plac...",COMPLETED,2006-08,Sanofi,"[Depressive Disorder, Depression, Depressive D...",[SSR149415],The primary efficacy endpoint is the change fr...,The main secondary endpoints are the changes f...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
7,NCT03595020,Probing Prefrontal Cortex Hemodynamic Alterati...,UNKNOWN,2017-02-16,Xijing Hospital,"[Depression, Depressive Disorder, Depressive D...",[],Hamilton Depression Scale-17(HAMD-17) score al...,Pittsburgh Sleep Quality Index（PSQI）;;; PSQI w...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
8,NCT04425473,Effect of Esketamine on Perioperative Depressi...,UNKNOWN,2021-02-19,Beijing Tiantan Hospital,[Depression],"[Esketamine, Normal saline]",Depressive symptoms remission;;; Remission is ...,Side-effects and Adverse events;;; All drug-re...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
9,NCT00489073,Duloxetine Versus Placebo in the Treatment of ...,COMPLETED,2002-11,Eli Lilly and Company,"[Fibromyalgia, Myofascial Pain Syndromes]","[duloxetine, placebo]",To assess efficacy of duloxetine 60 mg twice d...,To evaluate whether improvement in pain severi...,eligibilityCriteria: Inclusion Criteria:\n\n* ...


In [172]:
import requests
import pandas as pd
import json

df = pd.DataFrame(columns=[
    "nct_id", "Study_title", "Study_status", "Study_start_date",
    "Study_lead_sponsor_or_company", "Indications/Conditions", "Drug_names",
    "Primary_outcomes", "Secondary_outcome", "Eligibilty_criteria"
])

base_url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.term": "Drug",
    "query.cond": "Major Depressive Disorder",
    "pageSize": 100,
}

next_page_token = None
total_fetched = 0

while True:
    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print(f"Request failed with status code {response.status_code}")
        break

    data = response.json()
    studies = data.get("studies", [])
    print(f"Fetched {len(studies)} studies on this page")

    for i in studies:
        try:
            nct_id = i['protocolSection']['identificationModule']['nctId']
            official_title = i['protocolSection']['identificationModule'].get('officialTitle', "")
            overall_status = i['protocolSection']['statusModule'].get('overallStatus', "")
            study_start_date = i['protocolSection']['statusModule'].get('startDateStruct', {}).get('date', "")
            lead_sponsor = i['protocolSection']['sponsorCollaboratorsModule']['leadSponsor'].get('name', "")
            
            # Conditions
            conditions = [x['term'] for x in i['derivedSection'].get('conditionBrowseModule', {}).get('meshes', [])]

            # Drugs
            drug_names = []
            for z in i['protocolSection'].get('armsInterventionsModule', {}).get('interventions', []):
                if z.get('type') == 'DRUG':
                    drug_names.append(z.get('name'))

            # Primary Outcomes
            primary_outcomes = ";;; ".join(
                " ".join(str(v) for v in a.values()) for a in i['protocolSection'].get('outcomesModule', {}).get('primaryOutcomes', [])
            )

            # Secondary Outcomes
            secondary_outcomes = ";;; ".join(
                " ".join(str(v) for v in a.values()) for a in i['protocolSection'].get('outcomesModule', {}).get('secondaryOutcomes', [])
            )

            # Eligibility
            eligibility = ";;; ".join(
                f"{key}: {value}" for key, value in i['protocolSection'].get('eligibilityModule', {}).items()
            )

            df.loc[len(df)] = [
                nct_id, official_title, overall_status, study_start_date, lead_sponsor,
                conditions, drug_names, primary_outcomes, secondary_outcomes, eligibility
            ]

        except Exception as e:
            print(f"Error processing study: {e}")
            continue

    total_fetched += len(studies)

    # Stop if no more pages
    next_page_token = data.get("nextPageToken")
    if not next_page_token:
        break

print(f"\nTotal studies fetched: {total_fetched}")


Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 100 studies on this page
Fetched 96 studies on this page

Total studies fetched: 2196


In [175]:
df.tail(5)

,nct_id,Study_title,Study_status,Study_start_date,Study_lead_sponsor_or_company,Indications/Conditions,Drug_names,Primary_outcomes,Secondary_outcome,Eligibilty_criteria
2191,NCT00296777,Medication Treatment Following Neuropsychologi...,COMPLETED,2004-12,New York State Psychiatric Institute,"[Depression, Depressive Disorder, Depressive D...","[Escitalopram, Bupropion, Imipramine]",Hamilton Depression Scale (HAM-D) 7 mos.,Beck Depression Inventory (BDI) 7 mos.;;; Clin...,eligibilityCriteria: Inclusion Criteria:\n\n* ...
2192,NCT06775054,The Role of Cognitive Flexibility in Depressio...,COMPLETED,2023-03-15,Gulhane Training and Research Hospital,"[Depression, Depressive Disorder, Depressive D...",[],Assessing Cognitive Flexibility in TMS Treatme...,Tracking Depression Severity Changes Depressiv...,eligibilityCriteria: Inclusion Criteria:\n\nDi...
2193,NCT02874833,The Efficacy of Exercise in Depressive Episode...,COMPLETED,2016-07,Johannes Gutenberg University Mainz,"[Depression, Depressive Disorder, Bipolar Diso...",[],Severity of depressive symptoms - Quick Invent...,Adherence of internet-based concept Adherence ...,eligibilityCriteria: Inclusion Criteria:\n\n1....
2194,NCT02994433,NMDA Receptor Antagonist Nitrous Oxide Targets...,TERMINATED,2017-01-27,Washington University School of Medicine,"[Depressive Disorder, Depression, Depressive D...","[Nitrous Oxide, Placebo gas]",Comparison of functional connectivity between ...,,eligibilityCriteria: Inclusion Criteria:\n\n* ...
2195,NCT00789854,"A Randomised, 6-week, Multicentre, Open-label,...",COMPLETED,2008-11,AstraZeneca,"[Depression, Depressive Disorder, Depressive D...","[Quetiapine XR, Lithium carbonate, SSRI/Venlaf...",Change in Depressive Symptoms Between Randomis...,Depression Remission; Montgomery-Asberg Depres...,eligibilityCriteria: Inclusion Criteria:\n\n* ...


In [30]:
import requests

# Base URL and initial parameters
base_url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.cond": "Major Depressive Disorder",
    "pageSize": 100  # Max allowed
}

all_studies = []
next_page_token = None

print("Fetching studies...")

# Fetch all studies using pagination
while True:
    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print("Failed to fetch data. Status code:", response.status_code)
        break

    data = response.json()
    studies = data.get("studies", [])
    all_studies.extend(studies)

    print(f"Fetched {len(studies)} studies, total so far: {len(all_studies)}")

    # Check if there is another page
    next_page_token = data.get("nextPageToken")
    if not next_page_token:
        break

print(f"\nTotal studies fetched: {len(all_studies)}")




Fetching studies...
Fetched 100 studies, total so far: 100
Fetched 100 studies, total so far: 200
Fetched 100 studies, total so far: 300
Fetched 100 studies, total so far: 400
Fetched 100 studies, total so far: 500
Fetched 100 studies, total so far: 600
Fetched 100 studies, total so far: 700
Fetched 100 studies, total so far: 800
Fetched 100 studies, total so far: 900
Fetched 100 studies, total so far: 1000
Fetched 100 studies, total so far: 1100


KeyboardInterrupt: 

In [68]:
for i in range(0,10):
    print('-------------------------------------')
    study = all_studies[i]
    protocol_section = study.get("protocolSection", {})
    interventions_module = protocol_section.get("armsInterventionsModule", {}).get("interventions", [])

    #print(protocol_section)
    print(interventions_module)

    drug_interventions = [
        intervention.get("name")
        for intervention in interventions_module
        if intervention.get("type") == "Drug"]



# 🔍 Extract DRUG-type interventions
'''
print("\nExtracting interventions of type 'Drug'...")

for study in all_studies:
    protocol_section = study.get("protocolSection", {})
    interventions_module = protocol_section.get("armsInterventionsModule", {}).get("interventions", [])

    print(protocol_section)
    print(interventions_module)

    drug_interventions = [
        intervention.get("interventionName")
        for intervention in interventions_module
        if intervention.get("interventionType") == "Drug"
    ]

    if drug_interventions:
        nct_id = protocol_section.get("identificationModule", {}).get("nctId", "Unknown")
        print(f"\nStudy NCT ID: {nct_id}")
        print("Drug Interventions:")
        for drug in drug_interventions:
            print(f"  - {drug}")
'''

-------------------------------------
[]
-------------------------------------
[]
-------------------------------------
[{'type': 'DRUG', 'name': 'Escitalopram', 'description': '10-20 mg of escitalopram daily for eight weeks (10mg for the first 2 weeks, 20mg thereafter)', 'armGroupLabels': ['Escitalopram', 'Placebo'], 'otherNames': ['Lexapro']}]
-------------------------------------
[{'type': 'DIETARY_SUPPLEMENT', 'name': 'Zincas Forte', 'description': '25 mgZn/day + imipramine (100-200mg/day)', 'armGroupLabels': ['1'], 'otherNames': ['Zincas Forte (Farmapol, Poland)', 'Imipramin (Polfarma, Poland)']}, {'type': 'DIETARY_SUPPLEMENT', 'name': 'Placebo', 'description': 'placebo + imipramine (100-200mg/day)', 'armGroupLabels': ['2'], 'otherNames': ['Imipramin (Polfarma, Poland)']}]
-------------------------------------
[{'type': 'DRUG', 'name': 'Esketamine', 'description': 'Type= exact number, number= 0.20, unit= mg/kg, form= intravenous infusion, route= intravenous use. One single intrave

'\nprint("\nExtracting interventions of type \'Drug\'...")\n\nfor study in all_studies:\n    protocol_section = study.get("protocolSection", {})\n    interventions_module = protocol_section.get("armsInterventionsModule", {}).get("interventions", [])\n\n    print(protocol_section)\n    print(interventions_module)\n\n    drug_interventions = [\n        intervention.get("interventionName")\n        for intervention in interventions_module\n        if intervention.get("interventionType") == "Drug"\n    ]\n\n    if drug_interventions:\n        nct_id = protocol_section.get("identificationModule", {}).get("nctId", "Unknown")\n        print(f"\nStudy NCT ID: {nct_id}")\n        print("Drug Interventions:")\n        for drug in drug_interventions:\n            print(f"  - {drug}")\n'